# Seamless tiles via quantized RoPE frequencies

Text-to-image with FLUX 3 Image Klein, comparing two rotary position embeddings on the same prompt, seed and noise:

1. **original**: the stock `flux3.dit.rope_embed`.
2. **all-round quantized**: on the height and width axes (ids axis 1 and 2, 32 dims = 16 rotation planes each) every plane's
   frequency is moved to the nearest one that completes a whole number of cycles across the image edge, so the rotation
   is exactly periodic over the token grid. The other two axes (t and l) keep their original frequencies.
   This is the `round` rule of `rope_quant_viz.py` applied to every plane.

With a periodic position encoding the DiT cannot tell the left edge from the right edge (same for top/bottom), so the
generated image should tile without a seam. The prompt is assumed to be an already upsampled caption, so no upsampler is loaded.

`DiT.rope_embed` is `None` after construction; it must be assigned before `forward`, which is what each experiment does.

In [ ]:
import math
import os
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
os.environ.setdefault("WEIGHTS_DIR", str(REPO / "weights"))  # flux3.pipeline.WEIGHTS reads this at import time
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch import Tensor

import flux3.dit as dit_module
from flux3.autoencoder import latent_ids, latent_shape
from flux3.dit import AXES_DIM, IN_CHANNELS, THETA
from flux3.pipeline import IMAGE_ROPE_SCALE, MODELS, WEIGHTS, Pipeline, dit_path, pack, scaled_ids, to_pil
from flux3.sampling import denoise, denoise_flow_map, timesteps

MODEL = "distilled"  # "distilled": 16 steps, no CFG (flow map); "base": 50 steps, CFG 4
QUANT = ""           # "" (bf16), "fp8" or "nvfp4": the DiT variant under weights/<entry>/variants
DEVICE = "cuda"
WIDTH, HEIGHT = 1024, 1024
SEED = 1

## Models

Text encoder (Qwen3-VL-4B), autoencoder and DiT, all loaded once through `Pipeline`. Everything is frozen: gradients
would otherwise be tracked through the 16-50 DiT calls of the sampler and run out of GPU memory.

In [ ]:
entry = WEIGHTS / MODELS[MODEL]["dit"]  # every weights entry ships the text encoder and the autoencoder
pipe = Pipeline({MODEL: str(dit_path(MODEL, QUANT))}, str(entry / "text_encoder"), str(entry / "ae.safetensors"), device=DEVICE)
dit = pipe.dits[MODEL]

for module in (dit, pipe.autoencoder, pipe.text_encoder.model):
    module.eval().requires_grad_(False)
assert not any(p.requires_grad for m in (dit, pipe.autoencoder, pipe.text_encoder.model) for p in m.parameters())
print(f"DiT: flow_map={dit.flow_map}, rope_embed={dit.rope_embed}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## RoPE variants

The stock `rope(pos, dim, theta)` uses, per axis, the planes `omega_j = theta ** (-2j / dim)`. The image ids on the
h and w axes are `0, s, 2s, ..., (n-1)s` with `s = IMAGE_ROPE_SCALE`, so one edge of `n` latent tokens spans
`period = n * s` id units. A plane is periodic over the edge iff `omega * period = 2*pi*k` for an integer `k`;
`quantize_freqs` moves every plane to the nearest such frequency (`round`: `k = floor(cycles + 0.5)`, planes with
`k = 0` become blind to position).

`make_rope_embed` builds a drop-in replacement for `flux3.dit.rope_embed` from one frequency vector per axis; `None`
keeps the original frequencies of that axis.

In [ ]:
TWO_PI = 2 * math.pi


def axis_freqs(dim: int, theta: float = THETA) -> Tensor:
    # the original per-plane frequencies of one axis, as in flux3.dit.rope
    scale = torch.arange(0, dim, 2, dtype=torch.float64) / dim
    return 1.0 / (theta**scale)


def quantize_freqs(omega: Tensor, period: float, rule: str = "round") -> Tensor:
    # each frequency -> the nearest one whose rotation repeats after `period` id units: omega' = 2*pi*k / period
    cycles = omega * period / TWO_PI
    if rule == "round":
        k = torch.floor(cycles + 0.5)
    elif rule == "fundamental":
        k = torch.clamp(torch.floor(cycles + 0.5), min=1.0)
    elif rule == "keep":
        k = cycles
    else:
        raise ValueError(f"unknown rule {rule!r}")
    return k * TWO_PI / period


def rope_from_freqs(pos: Tensor, omega: Tensor) -> Tensor:
    # same as flux3.dit.rope, with an explicit frequency vector instead of (dim, theta)
    out = torch.einsum("...n,d->...nd", pos.to(omega.dtype), omega.to(pos.device))
    out = torch.stack([torch.cos(out), -torch.sin(out), torch.sin(out), torch.cos(out)], dim=-1)
    return out.reshape(*out.shape[:-1], 2, 2).float()


def make_rope_embed(freqs: list[Tensor | None]):
    # ids (B, L, 4) -> (B, 1, L, head_dim // 2, 2, 2); freqs[i] is the frequency vector of axis i, None = original
    assert len(freqs) == len(AXES_DIM)
    omegas = [axis_freqs(dim) if f is None else f for f, dim in zip(freqs, AXES_DIM)]

    def rope_embed(ids: Tensor) -> Tensor:
        with torch.autocast(device_type=ids.device.type, enabled=False):
            emb = torch.cat([rope_from_freqs(ids[..., i], omegas[i]) for i in range(len(AXES_DIM))], dim=-3)
        return emb.unsqueeze(1)

    return rope_embed


def quantized_rope_embed(h: int, w: int, tile: tuple[str, ...] = ("h", "w"), rule: str = "round"):
    # h, w: latent grid size; tile: which of the "h" / "w" axes are made periodic; t and l axes untouched
    freqs: list[Tensor | None] = [None] * len(AXES_DIM)
    if "h" in tile:
        freqs[1] = quantize_freqs(axis_freqs(AXES_DIM[1]), h * IMAGE_ROPE_SCALE, rule)
    if "w" in tile:
        freqs[2] = quantize_freqs(axis_freqs(AXES_DIM[2]), w * IMAGE_ROPE_SCALE, rule)
    return make_rope_embed(freqs)


original_rope_embed = dit_module.rope_embed

Two sanity checks: the rebuilt embedding with no quantization matches the stock function, and the quantized one is
exactly periodic (position `p` and `p + n` rotate identically).

In [ ]:
h_lat, w_lat = latent_shape(HEIGHT, WIDTH)
test_ids = scaled_ids(latent_ids(h_lat, w_lat, DEVICE))[None]
assert torch.allclose(make_rope_embed([None] * 4)(test_ids), original_rope_embed(test_ids), atol=1e-6)

quant = quantized_rope_embed(h_lat, w_lat)
shifted = test_ids.clone()
shifted[..., 1] += h_lat * IMAGE_ROPE_SCALE  # one full edge further down
shifted[..., 2] += w_lat * IMAGE_ROPE_SCALE  # one full edge further right
assert torch.allclose(quant(test_ids), quant(shifted), atol=1e-4)

omega = axis_freqs(AXES_DIM[1])
cycles = omega * h_lat * IMAGE_ROPE_SCALE / TWO_PI
print("h axis, cycles per edge, original :", [f"{c:.2f}" for c in cycles.tolist()])
print("h axis, cycles per edge, quantized:", [f"{c:.0f}" for c in (quantize_freqs(omega, h_lat * IMAGE_ROPE_SCALE) * h_lat * IMAGE_ROPE_SCALE / TWO_PI).tolist()])

## Sampling

Mirrors `Pipeline.generate` for text-to-image (no reference images), with the `rope_embed` chosen per call. Same seed
means the same starting noise for both variants.

In [ ]:
def generate(rope_embed, prompt: str, width: int = WIDTH, height: int = HEIGHT, steps: int | None = None,
             guidance: float | None = None, seed: int = SEED) -> Image.Image:
    dit.rope_embed = rope_embed
    steps = steps or MODELS[MODEL]["steps"]
    guidance = MODELS[MODEL]["guidance"] if guidance is None else guidance
    h, w = latent_shape(height, width)
    noise = torch.randn((h * w, IN_CHANNELS["image"]), generator=torch.Generator().manual_seed(seed))
    noise = noise.to(torch.bfloat16).float().to(DEVICE)
    ids = latent_ids(h, w, DEVICE)
    image = (noise, scaled_ids(ids))
    schedule = timesteps(steps, 16.0)
    if dit.flow_map:
        packed = pack([[pipe.text_encoder.encode(prompt), None, image]])
        tokens = denoise_flow_map(dit, packed, schedule)
    else:
        text = [pipe.text_encoder.encode(""), pipe.text_encoder.encode(prompt)]
        packed = pack([[text[0], None, image], [text[1], None, image]])
        tokens = denoise(dit, packed, schedule, guidance)
    return to_pil(pipe.autoencoder.decode(tokens, ids))


def tiled(image: Image.Image, reps: int = 2) -> Image.Image:
    canvas = Image.new("RGB", (image.width * reps, image.height * reps))
    for i in range(reps):
        for j in range(reps):
            canvas.paste(image, (j * image.width, i * image.height))
    return canvas


def seam_score(image: Image.Image) -> dict[str, float]:
    # mean |difference| across the wrap-around seams, against the mean over interior neighbours (1.0 = no visible seam)
    x = torch.from_numpy(np.asarray(image)).float()
    interior_w = (x[:, 1:] - x[:, :-1]).abs().mean().item()
    interior_h = (x[1:] - x[:-1]).abs().mean().item()
    wrap_w = (x[:, 0] - x[:, -1]).abs().mean().item()
    wrap_h = (x[0] - x[-1]).abs().mean().item()
    return {"w seam / interior": wrap_w / interior_w, "h seam / interior": wrap_h / interior_h}

## Prompt

An already upsampled caption (no upsampler in this notebook). A texture-like subject makes seams easy to judge.

In [ ]:
PROMPT = (
    "Top-down full-frame photograph of a dense forest floor texture, evenly lit by soft overcast daylight. "
    "The ground is covered edge to edge with a mix of small fallen autumn leaves in muted orange, ochre and brown, "
    "patches of bright green moss, thin fallen twigs and scattered acorns, with dark damp soil showing between them. "
    "The composition has no horizon, no single focal object and no empty areas: the pattern of leaves, moss and twigs "
    "continues uniformly across the whole frame with consistent scale and colour balance. Sharp focus throughout, "
    "natural matte colours, fine detail in the leaf veins and moss, no vignetting, no lens blur."
)

## Original vs all-round quantized RoPE

In [ ]:
h_lat, w_lat = latent_shape(HEIGHT, WIDTH)
variants = {
    "original": original_rope_embed,
    "quantized (h + w, all round)": quantized_rope_embed(h_lat, w_lat, tile=("h", "w"), rule="round"),
}

results = {}
for name, rope_embed in variants.items():
    results[name] = generate(rope_embed, PROMPT)
    print(f"{name:30s} {seam_score(results[name])}")

In [ ]:
fig, axes = plt.subplots(2, len(results), figsize=(7 * len(results), 14))
for col, (name, image) in enumerate(results.items()):
    axes[0, col].imshow(image)
    axes[0, col].set_title(name)
    axes[1, col].imshow(tiled(image))
    axes[1, col].set_title(f"{name}, tiled 2x2")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

out = REPO / "notebooks" / "outputs"
out.mkdir(exist_ok=True)
for name, image in results.items():
    stem = name.split(" ")[0]
    image.save(out / f"{stem}.png")
    tiled(image).save(out / f"{stem}_tiled.png")
print(f"saved to {out}")

## Tiling along one axis only

`tile=("w",)` makes only the width axis periodic (a horizontally repeating strip), `tile=("h",)` only the height axis.
Other per-plane rules from `rope_quant_viz.py` (`fundamental`, `keep`) are available through the `rule` argument.

In [ ]:
# image_w = generate(quantized_rope_embed(h_lat, w_lat, tile=("w",)), PROMPT)
# display(tiled(image_w))